In [44]:
from sklearn.preprocessing import MinMaxScaler
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM as lstm
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping
from keras import Input
import pmdarima as pm
from sklearn.metrics import mean_squared_error, mean_absolute_error
import pandas as pd
import numpy as np

In [45]:
def ARIMA(train, saisonalite, test=False, type_produit=False):
    if type_produit:
        train = np.log(train)
    # fit
    fit_arima = pm.auto_arima(train, seasonal=saisonalite)
    
    if type(test) != bool:
        # Évaluation du modèle
        n_test = len(test)
        predictions = fit_arima.predict(n_periods=n_test)

        # retour à l'échelle originale
        if type_produit:
            predictions = np.exp(predictions)
        
        rmse_lstm = np.sqrt(mean_squared_error(test, predictions))
        mae_lstm = mean_absolute_error(test, predictions)
        
        return fit_arima, rmse_lstm, mae_lstm
    return fit_arima

In [46]:
def create_sequences(data, taille_fenêtre):
    
    X = []
    y = []

    # on parcourt la série
    for i in range(len(data) - taille_fenêtre):
        
        # séquence d'entrée
        X.append(data[i:i+taille_fenêtre])
        
        # valeur à prédire
        y.append(data[i+taille_fenêtre])

    return np.array(X), np.array(y)

In [59]:
def LSTM(train, saisonalite, test=False, n_neurones = 32, optimizer="adam", loss="mse", metrics=["mae"], patience=10): # test = False si on
    # ne veut pas tester le modèle sur un ensmble de test

    # Le scaler transforme les données pour qu'elles soient entre 0 et 1.
    
    scaler = MinMaxScaler(feature_range=(0, 1))
    
    # On apprend le minimum et le maximum à partir du train seulement.
    # Cela évite d'utiliser des informations du futur.
    
    train_scaled = scaler.fit_transform(train)

    if type(test) != bool:
        # On applique exactement la même transformation au jeu de test.
        test_scaled = scaler.fit_transform(test)


    
    # Création des séquences

    L = saisonalite

    # création des séquences pour train
    X_train, y_train = create_sequences(train_scaled, L)

    if type(test) != bool:
        # création des séquences pour test
        X_test, y_test = create_sequences(test_scaled, L)

    # Les données sont transformées en séquences afin de permettre au LSTM
    # d'apprendre les dépendances temporelles.
    # 
    # On apprend au modèle : “Si les 12 derniers mois ressemblent à ceci → le mois suivant devrait être cela.”
    
    # Chaque entrée du modèle correspond à une fenêtre temporelle
    # contenant les L observations précédentes.
    #
    # La sortie correspond à la valeur suivante de la série.
    #
    # Cette transformation permet de reformuler le problème de prévision
    # en un problème supervisé adapté aux réseaux de neurones.

    # Construction du modèle LSTM

    # Création d'un modèle séquentiel
    model = Sequential()
    
    # Couche LSTM
    # 32 neurones = taille de la mémoire du réseau
    model = Sequential([
    Input(shape=(X_train.shape[1], 1)),
    lstm(n_neurones),
    Dense(1)
    ])

    model.compile(
    optimizer=optimizer,
    loss=loss,
    metrics=metrics
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=patience,
        restore_best_weights=True
    )

    model.fit(
    X_train,
    y_train,
    epochs= 80,  # nombre d’itérations d’apprentissage
     batch_size=16,    # nombre d'exemples traités à la fois
    validation_data=(X_test,y_test),
    callbacks=[early_stop]
    )

    if type(test) != bool:
        # Évaluation du modèle
        predictions = model.predict(X_test)
    
        # Remettre les valeurs à l’échelle originale; Comme on a normalisé les données, il faut annuler la normalisation.
        predictions = scaler.inverse_transform(predictions)
        y_test_real = scaler.inverse_transform(y_test.reshape(-1,1))
    
        rmse_lstm = np.sqrt(mean_squared_error(y_test_real, predictions))
        mae_lstm = mean_absolute_error(y_test_real, predictions)
        
        return model, rmse_lstm, mae_lstm, scaler
    return model, scaler
    

# ESSAI

In [48]:
# charger les données
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/airline-passengers.csv"
df = pd.read_csv(url)
# Conversion de la colonne de dates car avnt elles étaient des texte genre "01-2025"
df["Month"] = pd.to_datetime(df["Month"])

# On met la date comme index temporel pour dire à Python d'utiliser les dates comme axe du temps.
df.set_index("Month", inplace=True)
# 1.2 Division des données en apprentissage et test

# (80% des données d'entraînement)
train_size = int(len(df) * 0.8)

# Création des ensembles train et test
train = df.iloc[:train_size]
test = df.iloc[train_size:]

# Vérification des tailles
print("Taille train :", len(train))
print("Taille test :", len(test))

Taille train : 115
Taille test : 29


In [49]:
fit_arima, rmse_arima, mae_arima = ARIMA(train, 12, test=test, type_produit=True) # 12 mois par an

In [50]:
print(rmse_arima, mae_arima)
predictions = np.exp(fit_arima.predict(n_periods = len(test)))
predictions[:10]

91.08020294210444 80.9647405658993


1958-08-01    499.365473
1958-09-01    479.177118
1958-10-01    453.920047
1958-11-01    438.316155
1958-12-01    443.182558
1959-01-01    448.102990
1959-02-01    453.078051
1959-03-01    458.108348
1959-04-01    463.194494
1959-05-01    468.337108
Freq: MS, dtype: float64

In [60]:
fit_LSTM, rmse_LSTM, mae_LSTM, scaler = LSTM(train, 12, test=test) # 12 mois par an

Epoch 1/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 2s 73ms/step - loss: 0.0786 - mae: 0.2249 - val_loss: 0.0928 - val_mae: 0.2177
Epoch 2/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0205 - mae: 0.0952 - val_loss: 0.0632 - val_mae: 0.2086
Epoch 3/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0174 - mae: 0.1094 - val_loss: 0.0686 - val_mae: 0.2211
Epoch 4/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0157 - mae: 0.1051 - val_loss: 0.0638 - val_mae: 0.2095
Epoch 5/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0116 - mae: 0.0823 - val_loss: 0.0649 - val_mae: 0.2066
Epoch 6/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0117 - mae: 0.0811 - val_loss: 0.0647 - val_mae: 0.2074
Epoch 7/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0111 - mae: 0.0799 - val_loss: 0.0644 - val_mae: 0.2080
Epoch 8/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0108 - mae: 0.0796 - val_loss: 0.0641 - val_mae: 0.2089
Epoch 9/80
7/7 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 0.0108 - mae: 0.0801 - 

In [61]:
print(rmse_LSTM, mae_LSTM)

78.45592675054871 65.08037612017463


In [63]:
def forecast_lstm(model, train, scaler, saisonalite, n_steps):

    # scaler comme dans ton entraînement
    train_scaled = scaler.transform(train)

    # dernière fenêtre
    last_window = train_scaled[-saisonalite:]

    preds = []

    for _ in range(n_steps):

        X = last_window.reshape(1, saisonalite, 1)

        pred = model.predict(X, verbose=0)

        preds.append(pred[0,0])

        # mise à jour de la fenêtre
        last_window = np.vstack([last_window[1:], pred])

    preds = np.array(preds).reshape(-1,1)

    # retour à l'échelle originale
    preds = scaler.inverse_transform(preds)

    return preds

In [65]:
predictions = forecast_lstm(
    fit_LSTM,
    train,
    scaler,
    saisonalite=12,
    n_steps=10 # prédit les 10 prochaines périodes
)
predictions

array([[401.53998],
       [403.94733],
       [408.13583],
       [413.6844 ],
       [420.16882],
       [425.37558],
       [430.34012],
       [435.93256],
       [440.08435],
       [444.7251 ]], dtype=float32)